# auxA — Check Type and subType

List **all Type values, all subType values, and every Type/subType combination** in `dataset_wDR_mean_origIndex.h5`, with event counts and signed weight sums.

This is a census of the complete dataset: **no event selections, kinematic cuts, or ratio requirements**. Read only `Type`, `subType`, and `weight_kFact`. Labels are displayed with Python `repr`, so case differences, whitespace, byte strings, and missing values are visible.

The last table highlights labels containing `tt` or `top` and compares exact `Type == "ttV"` and `subType == "ttV"` matches. This identifies possible naming issues without assigning any process automatically.

In [1]:
%pip -q install numpy pandas tables


In [2]:

from pathlib import Path
import numpy as np
import pandas as pd
import tables
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    pass
else:
    drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/HZZ_OffShell")
DATA_PATH = PROJECT_DIR / "dataset_wDR_mean_origIndex.h5"
REPORT_DIR = PROJECT_DIR / "inspection" / "auxA"
REPORT_DIR.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


## Read just the three columns

The ntuple uses pandas fixed-format HDF5. Numeric columns can be sliced directly; the object block containing Type/subType must be deserialized together. This uses the same reading approach as the earlier notebooks, without importing their plotting or likelihood packages.

In [3]:
columns = ["Type", "subType", "weight_kFact"]
arrays = {}
with tables.open_file(DATA_PATH, mode="r") as handle:
    group = handle.get_node("/dataset")
    for block_id in range(group._v_attrs.nblocks):
        names = handle.get_node(group, f"block{block_id}_items").read().astype(str).tolist()
        wanted = [name for name in columns if name in names]
        if not wanted:
            continue
        node = handle.get_node(group, f"block{block_id}_values")
        if isinstance(node, tables.VLArray):
            block = node[0]
            for name in wanted:
                arrays[name] = block[:, names.index(name)].copy()
            del block
        else:
            for name in wanted:
                arrays[name] = node[:, names.index(name)]

missing = set(columns) - arrays.keys()
if missing:
    raise KeyError(f"Missing columns: {sorted(missing)}")
events = pd.DataFrame(arrays)
del arrays
print(f"Total events: {len(events):,}")


Total events: 9,540,480


## All labels and combinations

`events` counts every row, including rows with missing labels or nonfinite weights. `sumw_finite` sums the **signed finite** `weight_kFact` values; any nonfinite weights are counted separately. If a group has no finite weights, its sum is shown as NaN. Negative finite weights are retained in the sum.

The tables display every row and are also saved as CSV files. Quotes around labels are display notation, not added characters in the dataset.

In [4]:
w = events["weight_kFact"].to_numpy(dtype=np.float64)
finite_weight = np.isfinite(w)
events["sumw_finite"] = np.where(finite_weight, w, 0.0)
events["negative_weights"] = finite_weight & (w < 0)
events["nonfinite_weights"] = ~finite_weight


def summarize(keys):
    result = events.groupby(keys, dropna=False, sort=False).agg(
        events=("weight_kFact", "size"),
        sumw_finite=("sumw_finite", "sum"),
        negative_weights=("negative_weights", "sum"),
        nonfinite_weights=("nonfinite_weights", "sum"),
    ).reset_index()
    result.loc[result["nonfinite_weights"] == result["events"], "sumw_finite"] = np.nan
    for key in keys:
        result[key] = result[key].map(repr)
    return result


type_summary = summarize(["Type"])
subtype_summary = summarize(["subType"])
pair_summary = summarize(["Type", "subType"])

for title, filename, table in [
    ("All Type values", "types.csv", type_summary),
    ("All subType values", "subtypes.csv", subtype_summary),
    ("All Type / subType combinations", "type_subtype.csv", pair_summary),
]:
    print(f"\n{title}: {len(table)} entries")
    with pd.option_context("display.max_rows", None, "display.max_columns", None,
                           "display.max_colwidth", None, "display.float_format", "{:.8g}".format):
        display(table)
    table.to_csv(REPORT_DIR / filename, index=False)
    assert int(table["events"].sum()) == len(events)



All Type values: 7 entries


,Type,events,sumw_finite,negative_weights,nonfinite_weights
0,'S',2222114,36.855833,415,0
1,'SBI',1673988,359.28595,122,0
2,'B',1011799,368.40721,85,0
3,'qqZZ',2643586,1651.3179,488161,0
4,'EWB',535793,29.748743,421,0
5,'EWSBI',564555,25.549191,299,0
6,'EWSBI10',888645,54.917601,918,0



All subType values: 8 entries


,subType,events,sumw_finite,negative_weights,nonfinite_weights
0,'S',2222114,36.855833,415,0
1,'SBI',1673988,359.28595,122,0
2,'B',1011799,368.40721,85,0
3,'ttV',238545,75.192398,43860,0
4,'qqZZ',2405041,1576.1255,444301,0
5,'EWB',535793,29.748743,421,0
6,'EWSBI',564555,25.549191,299,0
7,'EWSBI10',888645,54.917601,918,0



All Type / subType combinations: 8 entries


,Type,subType,events,sumw_finite,negative_weights,nonfinite_weights
0,'S','S',2222114,36.855833,415,0
1,'SBI','SBI',1673988,359.28595,122,0
2,'B','B',1011799,368.40721,85,0
3,'qqZZ','ttV',238545,75.192398,43860,0
4,'qqZZ','qqZZ',2405041,1576.1255,444301,0
5,'EWB','EWB',535793,29.748743,421,0
6,'EWSBI','EWSBI',564555,25.549191,299,0
7,'EWSBI10','EWSBI10',888645,54.917601,918,0


## Check ttV labeling

The saved census shows that ttV events have **Type == "qqZZ" and subType == "ttV"**, while the qqZZ events have **subType == "qqZZ"**. The eight subType labels distinguish all physical processes directly.

Notebooks 01–05 therefore use **`subType == process_name` for every process**, including the reference and native yields. Notebook 05 includes `subType == "ttV"` separately; 01–04 exclude it. No additional Type condition is used.

The exact-match counts below show why selecting Type == "ttV" gives an empty sample. This auxiliary notebook still inspects both label columns to expose naming issues; the broad search does not change any analysis selection.

In [ ]:
exact_type = int(events["Type"].eq("ttV").sum())
exact_subtype = int(events["subType"].eq("ttV").sum())
print(f'Exact Type == "ttV": {exact_type:,} events')
print(f'Exact subType == "ttV": {exact_subtype:,} events')
if exact_type == 0:
    print('No exact Type == "ttV" rows; notebook 05 selects ttV using subType == "ttV".')

candidate = (pair_summary["Type"].str.contains("tt|top", case=False, na=False)
             | pair_summary["subType"].str.contains("tt|top", case=False, na=False))
ttv_labels = pair_summary.loc[candidate].reset_index(drop=True)
print('\nType/subType combinations containing "tt" or "top":')
if ttv_labels.empty:
    print("No matches; use the complete tables above.")
else:
    with pd.option_context("display.max_rows", None, "display.max_columns", None,
                           "display.max_colwidth", None, "display.float_format", "{:.8g}".format):
        display(ttv_labels)
ttv_labels.to_csv(REPORT_DIR / "ttv_label_candidates.csv", index=False)
print("\nSaved CSV tables:", REPORT_DIR)
